# HuggingFace Transformers — Pipeline Demo

A tour of the most common HuggingFace `pipeline` tasks using the latest `transformers` API.

> **Note:** Uses `device_map="auto"` — runs on GPU if available, CPU otherwise. No manual device index needed.

## Installation

In [13]:
!pip install -q transformers datasets accelerate torch

In [14]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")

Using: GPU


---
# NLP Tasks

## 1. Text Classification — Sentiment Analysis

`"sentiment-analysis"` is a deprecated alias. Use `"text-classification"` with an explicit model.

In [15]:
classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

result = classifier("I was so not happy with the last Mission Impossible Movie")
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9997795224189758}]


### Batch Classification

In [16]:
texts = [
    "I really like Autoencoders, best models for Anomaly Detection",
    "I am not sure if we can actually evaluate LLMs.",
    "PassiveAggressive is a linear regression model that so many people do not know about.",
    "I hate long meetings.",
]

results = classifier(texts)
for text, res in zip(texts, results):
    print(f"{res['label']:8s} ({res['score']:.2%})  {text}")

POSITIVE (99.79%)  I really like Autoencoders, best models for Anomaly Detection
NEGATIVE (99.95%)  I am not sure if we can actually evaluate LLMs.
NEGATIVE (99.61%)  PassiveAggressive is a linear regression model that so many people do not know about.
NEGATIVE (99.70%)  I hate long meetings.


### Multi-label Emotion Classification

In [17]:
emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    device=device,
)

texts = [
    "I really like Autoencoders, best models for Anomaly Detection",
    "I am not sure if we can actually evaluate LLMs.",
    "PassiveAggressive is a pretty funny name for a regression model.",
    "I hate long meetings.",
]

results = emotion_classifier(texts)
for text, res in zip(texts, results):
    print(f"{res['label']:20s} ({res['score']:.2%})  {text[:60]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


admiration           (74.07%)  I really like Autoencoders, best models for Anomaly Detectio
confusion            (90.49%)  I am not sure if we can actually evaluate LLMs.
amusement            (86.50%)  PassiveAggressive is a pretty funny name for a regression mo
anger                (77.20%)  I hate long meetings.


### Zero-Shot Classification

Classify text into any categories — no task-specific fine-tuning required.

In [18]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

result = zero_shot(
    "Everyday lots of LLM papers are published. Lots of them look very promising. "
    "I am not sure if we can actually evaluate LLMs. There is still lots to do.",
    candidate_labels=["optimistic", "pessimistic", "neutral"],
)
print(result)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'Everyday lots of LLM papers are published. Lots of them look very promising. I am not sure if we can actually evaluate LLMs. There is still lots to do.', 'labels': ['optimistic', 'neutral', 'pessimistic'], 'scores': [0.9071238040924072, 0.07738184183835983, 0.015494289807975292]}


### Zero-Shot: Why Label Wording Matters

The same text, three experiments:
1. **Vague labels** — broad words like "optimistic" get pulled toward any positive phrase
2. **Specific labels** — precise hypotheses score more accurately
3. **Sentence-level** — classify each sentence independently to see where the tone actually lives

In [19]:
text = (
    "Everyday lots of LLM papers are published. Lots of them look very promising. "
    "I am not sure if we can actually evaluate LLMs. There is still lots to do."
)

# ── 1. Vague labels (original) ───────────────────────────────────────────
r1 = zero_shot(text, candidate_labels=["optimistic", "pessimistic", "neutral"])
print("Vague labels:")
for label, score in zip(r1["labels"], r1["scores"]):
    print(f"  {label:15s} {score:.2%}")

# ── 2. Specific labels ───────────────────────────────────────────────────
r2 = zero_shot(text, candidate_labels=[
    "excited and confident about progress",
    "hopeful but uncertain about challenges",
    "doubtful and critical",
])
print("\nSpecific labels:")
for label, score in zip(r2["labels"], r2["scores"]):
    print(f"  {label:40s} {score:.2%}")

# ── 3. Sentence-level ────────────────────────────────────────────────────
sentences = [
    "Everyday lots of LLM papers are published.",
    "Lots of them look very promising.",
    "I am not sure if we can actually evaluate LLMs.",
    "There is still lots to do.",
]
print("\nSentence-level (optimistic / pessimistic / neutral):")
for sent in sentences:
    r = zero_shot(sent, candidate_labels=["optimistic", "pessimistic", "neutral"])
    top_label = r["labels"][0]
    top_score = r["scores"][0]
    print(f"  [{top_label:11s} {top_score:.0%}]  {sent}")

Vague labels:
  optimistic      90.71%
  neutral         7.74%
  pessimistic     1.55%

Specific labels:
  hopeful but uncertain about challenges   95.00%
  doubtful and critical                    3.74%
  excited and confident about progress     1.27%

Sentence-level (optimistic / pessimistic / neutral):
  [neutral     55%]  Everyday lots of LLM papers are published.
  [optimistic  99%]  Lots of them look very promising.
  [neutral     60%]  I am not sure if we can actually evaluate LLMs.
  [neutral     45%]  There is still lots to do.


## 2. Text Generation

In [20]:
text_generator = pipeline(
    "text-generation",
    model="distilbert/distilgpt2",
    device=device,
)

outputs = text_generator(
    "Today is a rainy day in London",
    max_new_tokens=50,
    num_return_sequences=2,
    truncation=True,
)
for i, out in enumerate(outputs):
    print(f"--- Sequence {i+1} ---")
    print(out["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Sequence 1 ---
Today is a rainy day in London. The main streets of London are the Hantsworths and the Woolwichs. The streets of London are the Hantsworths and the Woolwichs. The streets of London are the Hantsworths and the Woolwichs.
--- Sequence 2 ---
Today is a rainy day in London and is sometimes used to make a mess of things in the morning.



But what is the problem with that? It is a lot more than the problem of being at work, or at school. And what is it that can be


## 3. Question Answering

In [21]:
qa_model = pipeline(
    "question-answering",
    model="distilbert/distilbert-base-cased-distilled-squad",
    device=device,
)

result = qa_model(
    question="What is my job?",
    context="I am developing AI models with Python.",
)
print(result)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'score': 0.7823827266693115, 'start': 5, 'end': 25, 'answer': 'developing AI models'}


## 4. Named Entity Recognition (NER)

In [22]:
ner = pipeline(
    "token-classification",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
    device=device,
)

result = ner("Elon Musk founded SpaceX in Hawthorne, California.")
for entity in result:
    print(f"{entity['entity_group']:5s}  {entity['word']:20s}  score={entity['score']:.2%}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PER    Elon Musk             score=99.86%
ORG    SpaceX                score=99.89%
LOC    Hawthorne             score=99.00%
LOC    California            score=99.86%


## 5. Summarization

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "sshleifer/distilbart-cnn-12-6"
_sum_tokenizer = AutoTokenizer.from_pretrained(MODEL)
_sum_model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to("cuda" if torch.cuda.is_available() else "cpu")

article = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889
as the centerpiece of the 1889 World's Fair. The tower is 330 metres tall and was the world's tallest
man-made structure for 41 years until the Chrysler Building in New York City was completed in 1930.
More than 300 million people have visited the Eiffel Tower since it opened to the public.
"""

inputs = _sum_tokenizer(
    article, return_tensors="pt", max_length=1024, truncation=True
).to(_sum_model.device)

summary_ids = _sum_model.generate(
    inputs["input_ids"],
    max_new_tokens=60,
    min_new_tokens=20,
    num_beams=4,
    early_stopping=True,
)
print(_sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


 The Eiffel Tower was the centerpiece of the 1889 World's Fair . It is 330 metres tall and was the world's tallest man-made structure for 41 years . More than 300 million people have visited the tower since it opened to the public .


## 6. Translation

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

_tr_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
_tr_model     = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr").to("cuda" if torch.cuda.is_available() else "cpu")

text = "Machine learning is transforming every industry."

inputs = _tr_tokenizer(text, return_tensors="pt").to(_tr_model.device)
translated_ids = _tr_model.generate(**inputs, max_new_tokens=60)
print(_tr_tokenizer.decode(translated_ids[0], skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

L'apprentissage automatique transforme chaque industrie.


## 7. Fill-Mask

In [25]:
fill_mask = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased",
    device=device,
)

results = fill_mask("The capital of France is [MASK].")
for r in results:
    print(f"{r['score']:.2%}  {r['sequence']}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

41.68%  the capital of france is paris.
7.14%  the capital of france is lille.
6.34%  the capital of france is lyon.
4.44%  the capital of france is marseille.
3.03%  the capital of france is tours.


## 8. Sentence Similarity

`"sentence-similarity"` is not a built-in pipeline task. Use `"feature-extraction"` to get embeddings, then compute cosine similarity.

In [26]:
import torch
import torch.nn.functional as F

_embed_pipe = pipeline(
    "feature-extraction",
    model="sentence-transformers/all-MiniLM-L6-v2",
    device=device,
)

def sentence_similarity(s1: str, s2: str) -> float:
    e1 = torch.tensor(_embed_pipe(s1)[0]).mean(dim=0)
    e2 = torch.tensor(_embed_pipe(s2)[0]).mean(dim=0)
    return F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item()

pairs = [
    ("I love machine learning.", "Deep learning is fascinating."),
    ("I love machine learning.", "The weather today is sunny."),
]
for s1, s2 in pairs:
    score = sentence_similarity(s1, s2)
    print(f"Similarity: {score:.4f}")
    print(f"  A: {s1}")
    print(f"  B: {s2}")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Similarity: 0.6787
  A: I love machine learning.
  B: Deep learning is fascinating.
Similarity: -0.0323
  A: I love machine learning.
  B: The weather today is sunny.


---
# Computer Vision Tasks

## 9. Image Classification

In [34]:
image_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    device=device,
)

from datasets import load_dataset
from PIL import Image

dataset = load_dataset("huggingface/cats-image", split="test")
image = dataset[0]["image"]  # already a PIL Image
image.save("sample_dog.jpg")


results = image_classifier("sample_dog.jpg")
for r in results[:3]:
    print(f"{r['score']:.2%}  {r['label']}")


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


93.36%  Egyptian cat
4.71%  tabby, tabby cat
1.32%  tiger cat


## 10. Image Captioning

In [36]:
from transformers import BlipProcessor, BlipForConditionalGeneration

_blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
_blip_model     = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda" if torch.cuda.is_available() else "cpu")

inputs = _blip_processor(image, return_tensors="pt").to(_blip_model.device)
caption_ids = _blip_model.generate(**inputs, max_new_tokens=50)
print(_blip_processor.decode(caption_ids[0], skip_special_tokens=True))


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

two cats sleeping on a couch


---
# Tokenization

## Tokenizer Walkthrough — BERT

In [38]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

text = "I was so not happy with the Barbie Movie"

tokens = tokenizer.tokenize(text)
print("Tokens:    ", tokens)

input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Input IDs: ", input_ids)

encoded = tokenizer(text)
print("Encoded:   ", encoded)

decoded = tokenizer.decode(input_ids)
print("Decoded:   ", decoded)

Tokens:     ['i', 'was', 'so', 'not', 'happy', 'with', 'the', 'barbie', 'movie']
Input IDs:  [1045, 2001, 2061, 2025, 3407, 2007, 1996, 22635, 3185]
Encoded:    {'input_ids': [101, 1045, 2001, 2061, 2025, 3407, 2007, 1996, 22635, 3185, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Decoded:    i was so not happy with the barbie movie


### Cased vs Uncased

In [39]:
for model_name in ["google-bert/bert-base-uncased", "google-bert/bert-base-cased"]:
    tok = AutoTokenizer.from_pretrained(model_name)
    tokens = tok.tokenize(text)
    print(f"{model_name.split('/')[-1]:25s} → {tokens}")

bert-base-uncased         → ['i', 'was', 'so', 'not', 'happy', 'with', 'the', 'barbie', 'movie']


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

bert-base-cased           → ['I', 'was', 'so', 'not', 'happy', 'with', 'the', 'Barbie', 'Movie']


**token_type_ids** — distinguish segments in tasks like QA and sentence-pair classification. All zeros for single-sentence tasks.

**attention_mask** — `1` for real tokens, `0` for padding. Tells the model what to attend to.

**Padding** — sequences in a batch are padded to the same length so they can be processed in parallel.

---
# Fine-Tuning on IMDB

## Step 1 — Install Libraries

In [40]:
!pip install -q datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


## Step 2 — Load Dataset

In [41]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


## Step 3 — Tokenize

In [42]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized.set_format("torch")
print(tokenized)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


## Step 4 — Load Model

In [43]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 5 — Define Metrics

In [44]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

## Step 6 — Train with Trainer API

In [49]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./imdb-distilbert",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"].select(range(10000)),  # small subset for demo
    eval_dataset=tokenized["test"].select(range(10000)),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000002,0.000001,1.000000
2,0.000001,0.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1250, training_loss=2.5988274812698364e-05, metrics={'train_runtime': 178.5892, 'train_samples_per_second': 111.989, 'train_steps_per_second': 6.999, 'total_flos': 1324673986560000.0, 'train_loss': 2.5988274812698364e-05, 'epoch': 2.0})

## Step 7 — Evaluate

In [50]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 4.783868803315272e-07, 'eval_accuracy': 1.0, 'eval_runtime': 17.6277, 'eval_samples_per_second': 567.29, 'eval_steps_per_second': 17.756, 'epoch': 2.0}


## Step 8 — Inference with Fine-Tuned Model

In [51]:
fine_tuned = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=device,
)

reviews = [
    "This movie was absolutely fantastic! A masterpiece.",
    "Terrible film. I walked out after 20 minutes.",
    "It was okay, nothing special but not bad either.",
]

results = fine_tuned(reviews)
for review, res in zip(reviews, results):
    print(f"{res['label']:8s} ({res['score']:.2%})  {review}")

NEGATIVE (100.00%)  This movie was absolutely fantastic! A masterpiece.
NEGATIVE (100.00%)  Terrible film. I walked out after 20 minutes.
NEGATIVE (100.00%)  It was okay, nothing special but not bad either.


In [53]:
classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)


reviews = [
    "This movie was absolutely fantastic! A masterpiece.",
    "Terrible film. I walked out after 20 minutes.",
    "It was okay, nothing special but not bad either.",
]

results = classifier(reviews)
for review, res in zip(reviews, results):
    print(f"{res['label']:8s} ({res['score']:.2%})  {review}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

POSITIVE (99.99%)  This movie was absolutely fantastic! A masterpiece.
NEGATIVE (99.98%)  Terrible film. I walked out after 20 minutes.
POSITIVE (98.91%)  It was okay, nothing special but not bad either.
